# 🧠 Google Analytics Customer Revenue Prediction
## Phase 4 — Modeling, Validation & Submission Generation

> **Competition:** [Google Analytics Customer Revenue Prediction](https://www.kaggle.com/c/ga-customer-revenue-prediction)  
> **Notebook Environment:** Kaggle Notebooks  
> **Author:** Your Name

---

### 📌 Notebook Objectives

1. Load **4 datasets** — full features, important features (your idea), train split, val split
2. Train **multiple models** on each dataset
3. Use **temporal validation** — no random splits
4. Use **two-step strategy** — classify buyer first, then predict revenue amount
5. Generate **one submission file per model × dataset combination**

---

### 🗂️ Dataset Overview

| Dataset | Description | Source |
|---|---|---|
| `users_train.csv` | Full feature set — all users | Phase 3 output |
| `users_test.csv` | Full feature set — test users | Phase 3 output |
| `users_train_split.csv` | Temporal train portion | Phase 3 output |
| `users_val_split.csv` | Temporal validation portion | Phase 3 output |
| `YOUR_DATASET.csv` | Important features only (your idea 🎯) | Your dataset |
| `YOUR_DATASET_TEST.csv` | Important features — test users | Your dataset |

### 🤖 Models We Train

| Model | Strategy | Datasets |
|---|---|---|
| LightGBM Regressor | Direct log-revenue prediction | Full + Important features |
| XGBoost Regressor | Direct log-revenue prediction | Full + Important features |
| Two-Step LightGBM | Classify buyer → predict amount | Full + Important features |
| Two-Step XGBoost | Classify buyer → predict amount | Full + Important features |

---

## 📦 Section 1 — Imports

In [ ]:
import numpy as np
import pandas as pd
import os
import gc
import warnings
import json
from datetime import datetime

import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
import xgboost as xgb

from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.6f}'.format)

print('✅ Libraries imported')
print(f'📅 {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print(f'   LightGBM version : {lgb.__version__}')
print(f'   XGBoost  version : {xgb.__version__}')

---
## 📂 Section 2 — Paths

> **Edit the `YOUR_DATASET_*` paths** to point to your custom important-features dataset.

In [ ]:
# ── Phase 3 outputs ───────────────────────────────────────────────────────────
# Add the Phase 3 output as a dataset in Kaggle (Add Data → Your Work)
PHASE3_PATH = '/kaggle/input/your-phase3-dataset-name/'

# ── Your custom important-features dataset (your idea 🎯) ─────────────────────
# Edit these two paths after you add your dataset
YOUR_DATASET_TRAIN = '/kaggle/input/YOUR_DATASET_PATH/YOUR_TRAIN_FILE.csv'
YOUR_DATASET_TEST  = '/kaggle/input/YOUR_DATASET_PATH/YOUR_TEST_FILE.csv'

# ── Competition sample submission ─────────────────────────────────────────────
SAMPLE_SUB_PATH = '/kaggle/input/ga-customer-revenue-prediction/sample_submission_v2.csv'

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_PATH = '/kaggle/working/'

# ── Build full paths ──────────────────────────────────────────────────────────
TRAIN_FULL_PATH = os.path.join(PHASE3_PATH, 'users_train.csv')
TEST_FULL_PATH  = os.path.join(PHASE3_PATH, 'users_test.csv')
TRAIN_SPLIT     = os.path.join(PHASE3_PATH, 'users_train_split.csv')
VAL_SPLIT       = os.path.join(PHASE3_PATH, 'users_val_split.csv')

# ── Verify all files ──────────────────────────────────────────────────────────
all_paths = [
    (TRAIN_FULL_PATH,      'users_train.csv          (Phase 3)'),
    (TEST_FULL_PATH,       'users_test.csv           (Phase 3)'),
    (TRAIN_SPLIT,          'users_train_split.csv    (Phase 3)'),
    (VAL_SPLIT,            'users_val_split.csv      (Phase 3)'),
    (YOUR_DATASET_TRAIN,   'YOUR_TRAIN_FILE.csv      (Your idea 🎯)'),
    (YOUR_DATASET_TEST,    'YOUR_TEST_FILE.csv       (Your idea 🎯)'),
    (SAMPLE_SUB_PATH,      'sample_submission_v2.csv (Competition)'),
]

print('📁 File check:')
for path, name in all_paths:
    exists = os.path.exists(path)
    size   = f'{os.path.getsize(path)/1e6:.1f} MB' if exists else 'NOT FOUND — update path'
    print(f'  {"✅" if exists else "❌"}  {name:<45} {size}')

---
## 📥 Section 3 — Load All Datasets

In [ ]:
def load_user_dataset(path, name):
    df = pd.read_csv(path, dtype={'fullVisitorId': str}, low_memory=False)
    print(f'  ✅ {name:<35} shape: {df.shape}')
    return df

print('📂 Loading datasets...')
train_full  = load_user_dataset(TRAIN_FULL_PATH, 'users_train (full features)')
test_full   = load_user_dataset(TEST_FULL_PATH,  'users_test  (full features)')
train_split = load_user_dataset(TRAIN_SPLIT,     'users_train_split')
val_split   = load_user_dataset(VAL_SPLIT,       'users_val_split')

# ── Your important-features dataset ──────────────────────────────────────────
train_imp   = load_user_dataset(YOUR_DATASET_TRAIN, 'users_train (important feats 🎯)')
test_imp    = load_user_dataset(YOUR_DATASET_TEST,  'users_test  (important feats 🎯)')

# ── Sample submission ─────────────────────────────────────────────────────────
sample_sub = pd.read_csv(SAMPLE_SUB_PATH, dtype={'fullVisitorId': str})
print(f'  ✅ {"sample_submission_v2.csv":<35} shape: {sample_sub.shape}')

---
## ⚙️ Section 4 — Prepare Feature Matrices

Separate features (X), binary classification target (y_clf), and regression target (y_reg).

In [ ]:
NON_FEATURE_COLS = ['fullVisitorId', 'target', 'total_revenue', 'is_returning_user']

def prepare_matrices(df, name=''):
    """
    Returns X, y_reg (log revenue), y_clf (buyer vs non-buyer).
    """
    drop_cols = [c for c in NON_FEATURE_COLS if c in df.columns]
    X = df.drop(columns=drop_cols)

    y_reg = df['target'].values if 'target' in df.columns else None
    y_clf = (y_reg > 0).astype(int) if y_reg is not None else None

    # Ensure all numeric
    X = X.select_dtypes(include=[np.number])
    X = X.fillna(0)

    print(f'  {name:<35} X: {X.shape}  buyers: {y_clf.sum() if y_clf is not None else "N/A"}')
    return X, y_reg, y_clf


print('⚙️  Preparing feature matrices...')
X_train_full,  y_reg_full,  y_clf_full  = prepare_matrices(train_full,  'train_full')
X_test_full,   _,           _           = prepare_matrices(test_full,   'test_full')

X_tr_split,    y_reg_tr,    y_clf_tr    = prepare_matrices(train_split, 'train_split')
X_val_split,   y_reg_val,   y_clf_val   = prepare_matrices(val_split,   'val_split')

X_train_imp,   y_reg_imp,   y_clf_imp   = prepare_matrices(train_imp,   'train_imp (your idea 🎯)')
X_test_imp,    _,           _           = prepare_matrices(test_imp,    'test_imp  (your idea 🎯)')

# ── Align val split columns to train split columns ────────────────────────────
shared_cols = [c for c in X_tr_split.columns if c in X_val_split.columns]
X_tr_split  = X_tr_split[shared_cols]
X_val_split = X_val_split[shared_cols]

print(f'\n✅ All matrices ready')

---
## 📏 Section 5 — Evaluation Metric

Competition metric: **RMSE on log1p(revenue)** at user level.

In [ ]:
def rmse(y_true, y_pred):
    """Root Mean Squared Error — competition metric."""
    return np.sqrt(mean_squared_error(y_true, np.maximum(y_pred, 0)))


def evaluate(y_true, y_pred, label=''):
    score = rmse(y_true, y_pred)
    buyers_true = (y_true > 0).sum()
    buyers_pred = (np.array(y_pred) > 0.5).sum()
    print(f'  {label:<45} RMSE: {score:.6f}  |  buyers_true: {buyers_true}  buyers_pred: {buyers_pred}')
    return score


# Track all results for comparison
results = []

print('✅ Metric functions defined')

---
## 🤖 Section 6 — Model Configs

In [ ]:
# ── LightGBM params ───────────────────────────────────────────────────────────
LGBM_REG_PARAMS = dict(
    objective        = 'regression',
    metric           = 'rmse',
    n_estimators     = 1000,
    learning_rate    = 0.05,
    num_leaves       = 31,
    max_depth        = -1,
    min_child_samples= 20,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    reg_alpha        = 0.1,
    reg_lambda       = 0.1,
    random_state     = 42,
    n_jobs           = -1,
    verbose          = -1,
)

LGBM_CLF_PARAMS = dict(
    objective        = 'binary',
    metric           = 'auc',
    n_estimators     = 1000,
    learning_rate    = 0.05,
    num_leaves       = 31,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    reg_alpha        = 0.1,
    reg_lambda       = 0.1,
    random_state     = 42,
    n_jobs           = -1,
    verbose          = -1,
)

# ── XGBoost params ────────────────────────────────────────────────────────────
XGB_REG_PARAMS = dict(
    objective        = 'reg:squarederror',
    n_estimators     = 1000,
    learning_rate    = 0.05,
    max_depth        = 6,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    random_state     = 42,
    n_jobs           = -1,
    tree_method      = 'hist',
    verbosity        = 0,
)

XGB_CLF_PARAMS = dict(
    objective        = 'binary:logistic',
    n_estimators     = 1000,
    learning_rate    = 0.05,
    max_depth        = 6,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    random_state     = 42,
    n_jobs           = -1,
    tree_method      = 'hist',
    verbosity        = 0,
)

print('✅ Model configs defined')

---
## 🔁 Section 7 — Training Functions

### Strategy A: Direct Regression
Train directly on `log1p(revenue)` for all users. Simple and strong baseline.

### Strategy B: Two-Step (Classify → Regress)
- **Step 1:** Classify: is this user a buyer? (1/0)
- **Step 2:** For users predicted as buyers, predict how much revenue
- **Final:** `prediction = P(buyer) × predicted_revenue`

This handles zero-inflation explicitly and typically outperforms direct regression.

In [ ]:
def train_lgbm_direct(X_tr, y_tr, X_val, y_val, X_test, label):
    """
    Strategy A: LightGBM direct regression on log-revenue.
    """
    print(f'\n🟢 LightGBM Direct — {label}')

    model = lgb.LGBMRegressor(**LGBM_REG_PARAMS)
    model.fit(
        X_tr, y_tr,
        eval_set     = [(X_val, y_val)],
        callbacks    = [lgb.early_stopping(50, verbose=False),
                        lgb.log_evaluation(200)]
    )

    val_pred  = np.maximum(model.predict(X_val), 0)
    test_pred = np.maximum(model.predict(X_test), 0)
    score     = rmse(y_val, val_pred)

    print(f'   Best iteration : {model.best_iteration_}')
    print(f'   Validation RMSE: {score:.6f}')

    return model, val_pred, test_pred, score


def train_xgb_direct(X_tr, y_tr, X_val, y_val, X_test, label):
    """
    Strategy A: XGBoost direct regression on log-revenue.
    """
    print(f'\n🔵 XGBoost Direct — {label}')

    model = xgb.XGBRegressor(**XGB_REG_PARAMS)
    model.fit(
        X_tr, y_tr,
        eval_set          = [(X_val, y_val)],
        early_stopping_rounds = 50,
        verbose           = 200
    )

    val_pred  = np.maximum(model.predict(X_val), 0)
    test_pred = np.maximum(model.predict(X_test), 0)
    score     = rmse(y_val, val_pred)

    print(f'   Best iteration : {model.best_iteration}')
    print(f'   Validation RMSE: {score:.6f}')

    return model, val_pred, test_pred, score


def train_lgbm_twostep(X_tr, y_tr, y_tr_clf, X_val, y_val, y_val_clf, X_test, label):
    """
    Strategy B: LightGBM two-step.
    Step 1 → classify buyer
    Step 2 → regress revenue on buyers only
    Final  → P(buyer) × predicted_revenue
    """
    print(f'\n🟡 LightGBM Two-Step — {label}')

    # Step 1: Classifier
    clf = lgb.LGBMClassifier(**LGBM_CLF_PARAMS)
    clf.fit(
        X_tr, y_tr_clf,
        eval_set  = [(X_val, y_val_clf)],
        callbacks = [lgb.early_stopping(50, verbose=False),
                     lgb.log_evaluation(200)]
    )
    print(f'   Classifier best iter: {clf.best_iteration_}')

    # Step 2: Regressor on buyers only
    buyer_mask_tr = y_tr_clf == 1
    reg = lgb.LGBMRegressor(**LGBM_REG_PARAMS)
    reg.fit(
        X_tr[buyer_mask_tr], y_tr[buyer_mask_tr],
        eval_set  = [(X_val[y_val_clf == 1], y_val[y_val_clf == 1])],
        callbacks = [lgb.early_stopping(50, verbose=False),
                     lgb.log_evaluation(200)]
    )
    print(f'   Regressor best iter : {reg.best_iteration_}')

    # Final prediction: P(buyer) × revenue
    val_prob   = clf.predict_proba(X_val)[:, 1]
    val_amount = np.maximum(reg.predict(X_val), 0)
    val_pred   = val_prob * val_amount

    test_prob   = clf.predict_proba(X_test)[:, 1]
    test_amount = np.maximum(reg.predict(X_test), 0)
    test_pred   = test_prob * test_amount

    score = rmse(y_val, val_pred)
    print(f'   Validation RMSE: {score:.6f}')

    return (clf, reg), val_pred, test_pred, score


def train_xgb_twostep(X_tr, y_tr, y_tr_clf, X_val, y_val, y_val_clf, X_test, label):
    """
    Strategy B: XGBoost two-step.
    """
    print(f'\n🔴 XGBoost Two-Step — {label}')

    # Step 1: Classifier
    clf = xgb.XGBClassifier(**XGB_CLF_PARAMS)
    clf.fit(
        X_tr, y_tr_clf,
        eval_set              = [(X_val, y_val_clf)],
        early_stopping_rounds = 50,
        verbose               = 200
    )

    # Step 2: Regressor on buyers only
    buyer_mask_tr = y_tr_clf == 1
    reg = xgb.XGBRegressor(**XGB_REG_PARAMS)
    reg.fit(
        X_tr[buyer_mask_tr], y_tr[buyer_mask_tr],
        eval_set              = [(X_val[y_val_clf == 1], y_val[y_val_clf == 1])],
        early_stopping_rounds = 50,
        verbose               = 200
    )

    # Final prediction
    val_prob   = clf.predict_proba(X_val)[:, 1]
    val_amount = np.maximum(reg.predict(X_val), 0)
    val_pred   = val_prob * val_amount

    test_prob   = clf.predict_proba(X_test)[:, 1]
    test_amount = np.maximum(reg.predict(X_test), 0)
    test_pred   = test_prob * test_amount

    score = rmse(y_val, val_pred)
    print(f'   Validation RMSE: {score:.6f}')

    return (clf, reg), val_pred, test_pred, score


print('✅ All training functions defined')

---
## 🚀 Section 8 — Run All Experiments

We run every model × dataset combination and save a separate submission for each.

In [ ]:
def make_submission(test_df, test_preds, sample_sub, filename, label):
    """
    Creates a submission CSV matched to sample_submission_v2.csv format.
    Any user not in test predictions gets 0 revenue.
    """
    pred_df = pd.DataFrame({
        'fullVisitorId'      : test_df['fullVisitorId'].values,
        'PredictedLogRevenue': np.maximum(test_preds, 0)
    })

    # Aggregate per user (in case of duplicates)
    pred_df = pred_df.groupby('fullVisitorId')['PredictedLogRevenue'].mean().reset_index()

    # Merge with sample submission (keeps all required users)
    submission = sample_sub[['fullVisitorId']].merge(pred_df, on='fullVisitorId', how='left')
    submission['PredictedLogRevenue'] = submission['PredictedLogRevenue'].fillna(0)

    path = os.path.join(OUTPUT_PATH, filename)
    submission.to_csv(path, index=False)
    print(f'   💾 Saved: {filename}  ({os.path.getsize(path)/1e3:.1f} KB)  — {label}')
    return submission


print('✅ Submission function defined')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 1 — LightGBM Direct | Full Feature Dataset
# ═══════════════════════════════════════════════════════════════════════════════
_, val_pred, test_pred, score = train_lgbm_direct(
    X_tr_split, y_reg_tr,
    X_val_split, y_reg_val,
    X_test_full[shared_cols],
    label='Full Features'
)
results.append({'experiment': 'LGBM_Direct_FullFeatures', 'val_rmse': score})
make_submission(test_full, test_pred, sample_sub,
                'sub_01_lgbm_direct_full.csv', 'LightGBM Direct | Full Features')
gc.collect()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 2 — XGBoost Direct | Full Feature Dataset
# ═══════════════════════════════════════════════════════════════════════════════
_, val_pred, test_pred, score = train_xgb_direct(
    X_tr_split, y_reg_tr,
    X_val_split, y_reg_val,
    X_test_full[shared_cols],
    label='Full Features'
)
results.append({'experiment': 'XGB_Direct_FullFeatures', 'val_rmse': score})
make_submission(test_full, test_pred, sample_sub,
                'sub_02_xgb_direct_full.csv', 'XGBoost Direct | Full Features')
gc.collect()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 3 — LightGBM Two-Step | Full Feature Dataset
# ═══════════════════════════════════════════════════════════════════════════════
_, val_pred, test_pred, score = train_lgbm_twostep(
    X_tr_split, y_reg_tr, y_clf_tr,
    X_val_split, y_reg_val, y_clf_val,
    X_test_full[shared_cols],
    label='Full Features'
)
results.append({'experiment': 'LGBM_TwoStep_FullFeatures', 'val_rmse': score})
make_submission(test_full, test_pred, sample_sub,
                'sub_03_lgbm_twostep_full.csv', 'LightGBM Two-Step | Full Features')
gc.collect()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 4 — XGBoost Two-Step | Full Feature Dataset
# ═══════════════════════════════════════════════════════════════════════════════
_, val_pred, test_pred, score = train_xgb_twostep(
    X_tr_split, y_reg_tr, y_clf_tr,
    X_val_split, y_reg_val, y_clf_val,
    X_test_full[shared_cols],
    label='Full Features'
)
results.append({'experiment': 'XGB_TwoStep_FullFeatures', 'val_rmse': score})
make_submission(test_full, test_pred, sample_sub,
                'sub_04_xgb_twostep_full.csv', 'XGBoost Two-Step | Full Features')
gc.collect()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 5 — LightGBM Direct | Your Important Features Dataset 🎯
# ═══════════════════════════════════════════════════════════════════════════════

# Build val split from your important features dataset
# (assuming it has the same fullVisitorId and target columns)
imp_val_ids  = val_split['fullVisitorId'].values
imp_tr_ids   = train_split['fullVisitorId'].values

train_imp_tr  = train_imp[train_imp['fullVisitorId'].isin(imp_tr_ids)]
train_imp_val = train_imp[train_imp['fullVisitorId'].isin(imp_val_ids)]

Xi_tr,  yi_reg_tr,  yi_clf_tr  = prepare_matrices(train_imp_tr,  'imp_train')
Xi_val, yi_reg_val, yi_clf_val = prepare_matrices(train_imp_val, 'imp_val')
Xi_test, _, _ = prepare_matrices(test_imp, 'imp_test')

shared_imp = [c for c in Xi_tr.columns if c in Xi_val.columns and c in Xi_test.columns]
Xi_tr   = Xi_tr[shared_imp]
Xi_val  = Xi_val[shared_imp]
Xi_test = Xi_test[shared_imp]

_, val_pred, test_pred, score = train_lgbm_direct(
    Xi_tr, yi_reg_tr,
    Xi_val, yi_reg_val,
    Xi_test,
    label='Important Features 🎯'
)
results.append({'experiment': 'LGBM_Direct_ImportantFeatures', 'val_rmse': score})
make_submission(test_imp, test_pred, sample_sub,
                'sub_05_lgbm_direct_imp.csv', 'LightGBM Direct | Important Features 🎯')
gc.collect()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 6 — XGBoost Direct | Your Important Features Dataset 🎯
# ═══════════════════════════════════════════════════════════════════════════════
_, val_pred, test_pred, score = train_xgb_direct(
    Xi_tr, yi_reg_tr,
    Xi_val, yi_reg_val,
    Xi_test,
    label='Important Features 🎯'
)
results.append({'experiment': 'XGB_Direct_ImportantFeatures', 'val_rmse': score})
make_submission(test_imp, test_pred, sample_sub,
                'sub_06_xgb_direct_imp.csv', 'XGBoost Direct | Important Features 🎯')
gc.collect()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 7 — LightGBM Two-Step | Your Important Features Dataset 🎯
# ═══════════════════════════════════════════════════════════════════════════════
_, val_pred, test_pred, score = train_lgbm_twostep(
    Xi_tr, yi_reg_tr, yi_clf_tr,
    Xi_val, yi_reg_val, yi_clf_val,
    Xi_test,
    label='Important Features 🎯'
)
results.append({'experiment': 'LGBM_TwoStep_ImportantFeatures', 'val_rmse': score})
make_submission(test_imp, test_pred, sample_sub,
                'sub_07_lgbm_twostep_imp.csv', 'LightGBM Two-Step | Important Features 🎯')
gc.collect()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 8 — XGBoost Two-Step | Your Important Features Dataset 🎯
# ═══════════════════════════════════════════════════════════════════════════════
_, val_pred, test_pred, score = train_xgb_twostep(
    Xi_tr, yi_reg_tr, yi_clf_tr,
    Xi_val, yi_reg_val, yi_clf_val,
    Xi_test,
    label='Important Features 🎯'
)
results.append({'experiment': 'XGB_TwoStep_ImportantFeatures', 'val_rmse': score})
make_submission(test_imp, test_pred, sample_sub,
                'sub_08_xgb_twostep_imp.csv', 'XGBoost Two-Step | Important Features 🎯')
gc.collect()

---
## 📊 Section 9 — Results Comparison

In [ ]:
results_df = pd.DataFrame(results).sort_values('val_rmse')
results_df['rank'] = range(1, len(results_df) + 1)

print('\n🏆 All Experiment Results (sorted by Validation RMSE):')
print('='*65)
print(f'{"Rank":<6} {"Experiment":<40} {"Val RMSE"}')
print('-'*65)
for _, row in results_df.iterrows():
    marker = ' ← BEST' if row['rank'] == 1 else ''
    print(f"{int(row['rank']):<6} {row['experiment']:<40} {row['val_rmse']:.6f}{marker}")
print('='*65)

# Save results table
results_df.to_csv(OUTPUT_PATH + 'experiment_results.csv', index=False)
print('\n✅ Results saved to experiment_results.csv')

In [ ]:
# ── Visual comparison ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#2ca02c' if i == 0 else '#1f77b4' if 'LGBM' in e else '#d62728'
          for i, e in enumerate(results_df['experiment'])]

bars = ax.barh(results_df['experiment'][::-1],
               results_df['val_rmse'][::-1],
               color=colors[::-1])

for bar, val in zip(bars, results_df['val_rmse'][::-1]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.5f}', va='center', fontsize=9)

ax.set_title('All Experiments — Validation RMSE (lower is better)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Validation RMSE')
ax.axvline(results_df['val_rmse'].min(), color='gold', linestyle='--',
           linewidth=2, label='Best score')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_PATH + 'plot_10_experiment_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 📋 Section 10 — Phase 4 Summary

| Step | Action | Status |
|---|---|---|
| Load all datasets | Phase 3 outputs + your important features dataset | ✅ |
| Feature matrices | X, y_reg, y_clf for each dataset | ✅ |
| Temporal split | Train < Dec 2017 / Val ≥ Dec 2017 | ✅ |
| LightGBM Direct × 2 datasets | Experiments 1, 5 | ✅ |
| XGBoost Direct × 2 datasets | Experiments 2, 6 | ✅ |
| LightGBM Two-Step × 2 datasets | Experiments 3, 7 | ✅ |
| XGBoost Two-Step × 2 datasets | Experiments 4, 8 | ✅ |
| 8 submission files | sub_01 through sub_08 | ✅ |
| Results comparison table + chart | experiment_results.csv | ✅ |

---

### 📤 Submission Files Generated

| File | Model | Dataset |
|---|---|---|
| `sub_01_lgbm_direct_full.csv` | LightGBM Direct | Full features |
| `sub_02_xgb_direct_full.csv` | XGBoost Direct | Full features |
| `sub_03_lgbm_twostep_full.csv` | LightGBM Two-Step | Full features |
| `sub_04_xgb_twostep_full.csv` | XGBoost Two-Step | Full features |
| `sub_05_lgbm_direct_imp.csv` | LightGBM Direct | Your important features 🎯 |
| `sub_06_xgb_direct_imp.csv` | XGBoost Direct | Your important features 🎯 |
| `sub_07_lgbm_twostep_imp.csv` | LightGBM Two-Step | Your important features 🎯 |
| `sub_08_xgb_twostep_imp.csv` | XGBoost Two-Step | Your important features 🎯 |

Submit all 8 to Kaggle and compare public leaderboard scores against validation RMSE.

---

### ➡️ Next: Phase 5 — Ensemble & Final Submission

Blend the best-performing submissions using weighted averaging.  
A blend of the best two-step model + best direct model typically outperforms either alone.